# Nature-style strategy figures v5

Updated to use larger typography, area-based wording, no chart grid lines, and a half-width Supplementary Fig. 1.

In [ ]:

import os
import math
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

# -----------------------------
# Paths and data
# -----------------------------
PKG = Path("/mnt/data/nature_strategy_package_v5")
PKG.mkdir(parents=True, exist_ok=True)

overall = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_overall_summary.csv")
selected = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_selected_solutions.csv")
province = pd.read_csv("/mnt/data/pareto_strategy_analysis/pareto_strategy_province_summary.csv")

# -----------------------------
# Typography and palette
# -----------------------------
FS_TICK = 14
FS_SMALL = 12
FS_ANNOT = 13
FS_AXIS = 17
FS_PANEL_TITLE = 18
FS_PANEL_LABEL = 21
FS_FIG_TITLE = 20
FS_LEGEND = 11
FS_SUP_TITLE = 14

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "#4A4A4A",
    "axes.labelcolor": "#333333",
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "text.color": "#2F2F2F",
})

COLORS = {
    "Target-first": "#C67C6B",
    "Maximum mitigation": "#8E6558",
    "Cost-efficiency-first": "#D7A38C",
    "Balanced compromise": "#B7A08D",
}
GREY = "#6E6A67"
LIGHT_GREY = "#CFC7C1"
MID_GREY = "#A89D96"

MAP_EN = {
    "达标优先": "Target-first",
    "极限减灾": "Maximum mitigation",
    "性价比优先": "Cost-efficiency-first",
    "平衡折中": "Balanced compromise",
}
ORDER_CN = ["达标优先", "极限减灾", "性价比优先", "平衡折中"]
ORDER = [MAP_EN[x] for x in ORDER_CN]

overall["strategy_en"] = overall["strategy"].map(MAP_EN)
overall = overall.set_index("strategy_en").loc[ORDER].reset_index()

selected["strategy_en"] = selected["strategy"].map(MAP_EN)
selected["attain"] = selected["RR"] >= 0.35

province["strategy_en"] = province["strategy"].map(MAP_EN)

att = (
    selected.pivot(index="row_id", columns="strategy_en", values="attain")
    .fillna(False)
    .astype(bool)
)
att = att[ORDER]
combo_counts = att.value_counts().reset_index(name="count")
combo_counts["fraction"] = combo_counts["count"] / len(att)
combo_counts = combo_counts.sort_values("count", ascending=False).reset_index(drop=True)

def combo_to_label(row):
    labs = [k for k in ORDER if row[k]]
    return "None" if len(labs) == 0 else " + ".join(labs)

combo_counts["label"] = combo_counts.apply(combo_to_label, axis=1)

overall.head()


In [ ]:

# -----------------------------
# Helpers
# -----------------------------
def style_ax(ax):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=FS_TICK, length=3.5, width=0.8)

def add_panel_label(ax, label, title):
    ax.text(-0.10, 1.06, label, transform=ax.transAxes, fontsize=FS_PANEL_LABEL,
            fontweight="bold", va="bottom", ha="left")
    ax.text(-0.005, 1.06, title, transform=ax.transAxes, fontsize=FS_PANEL_TITLE,
            fontweight="bold", va="bottom", ha="left")

def barycentric_to_xy(isa, wtd, lai):
    h = math.sqrt(3) / 2.0
    x = wtd + 0.5 * lai
    y = h * lai
    return x, y

def draw_ternary_axes(ax):
    h = math.sqrt(3) / 2.0
    tri = np.array([[0, 0], [1, 0], [0.5, h], [0, 0]])
    ax.plot(tri[:, 0], tri[:, 1], color=GREY, lw=1.1)
    for v in [0.2, 0.4, 0.6, 0.8]:
        p1 = barycentric_to_xy(v, 1 - v, 0)
        p2 = barycentric_to_xy(v, 0, 1 - v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)
        p1 = barycentric_to_xy(1 - v, v, 0)
        p2 = barycentric_to_xy(0, v, 1 - v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)
        p1 = barycentric_to_xy(1 - v, 0, v)
        p2 = barycentric_to_xy(0, 1 - v, v)
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color="#ECE7E2", lw=0.85, zorder=0)

    ax.text(-0.03, -0.075, "ISA cost share", fontsize=FS_SMALL, ha="left", va="top")
    ax.text(1.03, -0.075, "WTD cost share", fontsize=FS_SMALL, ha="right", va="top")
    ax.text(0.50, math.sqrt(3) / 2.0 + 0.085, "LAI cost share", fontsize=FS_SMALL, ha="center", va="bottom")

    for v in [0.2, 0.4, 0.6, 0.8]:
        x, y = barycentric_to_xy(v, 1-v, 0)
        ax.text(x-0.022, y+0.024, f"{int(v*100)}%", fontsize=10, color=GREY, ha="right")
        x, y = barycentric_to_xy(0, v, 1-v)
        ax.text(x+0.022, y+0.024, f"{int(v*100)}%", fontsize=10, color=GREY, ha="left")
        x, y = barycentric_to_xy(1-v, 0, v)
        ax.text(x, y+0.038, f"{int(v*100)}%", fontsize=10, color=GREY, ha="center")

    ax.set_xlim(-0.09, 1.09)
    ax.set_ylim(-0.12, math.sqrt(3) / 2.0 + 0.13)
    ax.set_aspect("equal")
    ax.axis("off")


In [ ]:

# -----------------------------
# Panels
# -----------------------------
def panel_a_strategy_landscape(ax):
    style_ax(ax)
    add_panel_label(ax, "a", "Strategy landscape")

    for _, row in overall.iterrows():
        ax.scatter(
            row["mean_cost_total_million_usd"],
            row["mean_RR"],
            s=2300 * row["attainment_rate_rr_ge_0_35"] + 170,
            color=COLORS[row["strategy_en"]],
            edgecolor="white",
            linewidth=1.7,
            zorder=3,
            alpha=0.98,
        )

    offsets = {
        "Target-first": (8, -0.028),
        "Maximum mitigation": (-92, 0.022),
        "Cost-efficiency-first": (11, 0.052),
        "Balanced compromise": (-72, -0.050),
    }

    for _, row in overall.iterrows():
        dx, dy = offsets[row["strategy_en"]]
        label = (
            f"{row['strategy_en']}\n"
            f"Risk level change rate = {row['mean_RR']:.3f}\n"
            f"Mean cost per 100 km² = {row['mean_cost_total_million_usd']:.2f} M USD\n"
            f"Threshold-reaching area fraction = {row['attainment_rate_rr_ge_0_35']*100:.2f}%"
        )
        ax.annotate(
            label,
            xy=(row["mean_cost_total_million_usd"], row["mean_RR"]),
            xytext=(row["mean_cost_total_million_usd"] + dx, row["mean_RR"] + dy),
            fontsize=FS_ANNOT,
            ha="left" if dx >= 0 else "right",
            va="center",
            arrowprops=dict(arrowstyle="-", color=GREY, lw=1.0),
            bbox=dict(boxstyle="round,pad=0.28", fc="white", ec=LIGHT_GREY, lw=0.9),
            zorder=4,
        )

    tf = overall.loc[overall["strategy_en"] == "Target-first"].iloc[0]
    mm = overall.loc[overall["strategy_en"] == "Maximum mitigation"].iloc[0]
    ax.annotate(
        "Target-first reaches the same threshold-oriented area fraction\nas maximum mitigation, but at 56.24% lower mean cost.",
        xy=((tf["mean_cost_total_million_usd"] + mm["mean_cost_total_million_usd"]) / 2,
            (tf["mean_RR"] + mm["mean_RR"]) / 2),
        xytext=(119, 0.602),
        fontsize=FS_SMALL,
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=LIGHT_GREY, lw=0.9),
    )

    ax.set_xlabel("Mean cost per 100 km² in 20 years (million USD)", fontsize=FS_AXIS)
    ax.set_ylabel("Risk level change rate", fontsize=FS_AXIS)
    ax.set_xlim(0, 240)
    ax.set_ylim(0.22, 0.615)

    legend_x, legend_y = 0.055, 0.885
    ax.text(
        legend_x, legend_y + 0.095,
        "Bubble area encodes the fraction of 100 km² areas\nthat reach the 0.35 threshold.",
        transform=ax.transAxes, fontsize=FS_SMALL, ha="left", va="bottom"
    )
    for i, frac in enumerate([0.32, 0.50, 0.55]):
        ax.scatter(
            [legend_x + i * 0.155],
            [legend_y],
            s=2300 * frac + 170,
            color="none",
            edgecolor=GREY,
            linewidth=1.0,
            transform=ax.transAxes,
            clip_on=False,
        )
        ax.text(
            legend_x + i * 0.155,
            legend_y - 0.09,
            f"{frac*100:.0f}%",
            transform=ax.transAxes,
            fontsize=10.5, ha="center", va="top"
        )

def panel_b_area_ridgeline(ax):
    add_panel_label(ax, "b", "Area-level distributions")
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(labelsize=FS_TICK, length=3.5, width=0.8)

    xs = np.linspace(0, 1, 400)
    base_levels = np.arange(len(ORDER))[::-1]
    for level, strategy in zip(base_levels, ORDER):
        data = selected.loc[selected["strategy_en"] == strategy, "RR"].to_numpy()
        kde = gaussian_kde(data, bw_method=0.08)
        ys = kde(xs)
        ys = ys / ys.max() * 0.84
        ax.fill_between(xs, level, level + ys, color=COLORS[strategy], alpha=0.78, linewidth=0)
        ax.plot(xs, level + ys, color=COLORS[strategy], lw=1.8)
        mean_v = data.mean()
        med_v = np.median(data)
        ax.scatter([mean_v], [level + 0.45], s=54, facecolor="white", edgecolor=GREY, linewidth=1.15, zorder=5)
        ax.scatter([med_v], [level + 0.45], s=44, facecolor=GREY, edgecolor="white", linewidth=0.7, zorder=5)
        ax.text(-0.035, level + 0.16, strategy, ha="right", va="center", fontsize=FS_SMALL)

    ax.axvline(0.35, color=GREY, lw=1.25, ls=(0, (5, 3)))
    ax.text(
        0.35, len(ORDER)-0.13,
        "Target threshold = 0.35",
        fontsize=FS_SMALL, ha="center", va="bottom",
        bbox=dict(boxstyle="round,pad=0.22", fc="white", ec=LIGHT_GREY, lw=0.8)
    )

    handles = [
        Line2D([0], [0], marker="o", markersize=7.5, markerfacecolor="white", markeredgecolor=GREY,
               linestyle="None", label="Mean"),
        Line2D([0], [0], marker="o", markersize=6.2, markerfacecolor=GREY, markeredgecolor="white",
               linestyle="None", label="Median"),
        Line2D([0], [0], color=GREY, lw=1.2, ls=(0, (5, 3)), label="Threshold"),
    ]
    ax.legend(handles=handles, loc="lower right", frameon=False, fontsize=FS_LEGEND)
    ax.set_xlabel("Risk level change rate", fontsize=FS_AXIS)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.22, len(ORDER) + 0.25)
    ax.set_yticks([])

def panel_c_attainment_intersections(fig, gs_cell):
    sub = gs_cell.subgridspec(2, 1, height_ratios=[2.25, 1.25], hspace=0.035)
    ax_bar = fig.add_subplot(sub[0])
    ax_mat = fig.add_subplot(sub[1], sharex=ax_bar)

    cc = combo_counts.copy()
    x = np.arange(len(cc))

    style_ax(ax_bar)
    add_panel_label(ax_bar, "c", "Threshold-reaching area intersections")
    bars = ax_bar.bar(x, cc["fraction"] * 100, color="#CDBFB5", edgecolor="white", linewidth=1.0)
    ax_bar.set_ylabel("Share of total area (%)", fontsize=FS_AXIS)
    ax_bar.set_xticks([])
    ax_bar.set_xlim(-0.6, len(cc) - 0.4)
    ax_bar.set_ylim(0, max(cc["fraction"] * 100) * 1.23)

    for bar, frac, count in zip(bars, cc["fraction"], cc["count"]):
        ax_bar.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1.2,
            f"{frac*100:.2f}%\n(n={count:,})",
            ha="center", va="bottom", fontsize=10.3
        )

    ax_bar.text(
        0.01, 0.97,
        "Only 54.63% of the analysed 100 km² areas reached the threshold\nunder at least one decision rule.",
        transform=ax_bar.transAxes,
        ha="left", va="top", fontsize=FS_SMALL,
        bbox=dict(boxstyle="round,pad=0.28", fc="white", ec=LIGHT_GREY, lw=0.8),
    )

    ax_mat.set_facecolor("white")
    for spine in ax_mat.spines.values():
        spine.set_visible(False)
    ax_mat.set_ylim(-0.5, len(ORDER)-0.5)
    ax_mat.set_yticks(np.arange(len(ORDER)))
    ax_mat.set_yticklabels(ORDER, fontsize=FS_SMALL)
    ax_mat.invert_yaxis()
    ax_mat.set_xticks(x)
    ax_mat.set_xticklabels([""] * len(cc))
    ax_mat.tick_params(axis="x", length=0)

    for i in range(len(cc)):
        ys = []
        for j, strategy in enumerate(ORDER):
            active = bool(cc.loc[i, strategy])
            if active:
                ax_mat.scatter(i, j, s=88, color=COLORS[strategy], edgecolor="white", linewidth=0.9, zorder=3)
                ys.append(j)
            else:
                ax_mat.scatter(i, j, s=30, color="#DDD7D1", edgecolor="none", zorder=2)
        if len(ys) >= 2:
            ax_mat.plot([i, i], [min(ys), max(ys)], color=GREY, lw=1.1, zorder=1)

    for i, lbl in enumerate(cc["label"]):
        ax_mat.text(i, len(ORDER)-0.06, lbl.replace(" + ", "\n+\n"), ha="center", va="top", fontsize=9.2)

def panel_d_cost_ternary(ax):
    add_panel_label(ax, "d", "Cost composition")
    draw_ternary_axes(ax)

    for _, row in overall.iterrows():
        isa = row["mean_cost_ISA_million_usd"] / row["mean_cost_total_million_usd"]
        wtd = row["mean_cost_WT_million_usd"] / row["mean_cost_total_million_usd"]
        lai = row["mean_cost_LAI_million_usd"] / row["mean_cost_total_million_usd"]
        x, y = barycentric_to_xy(isa, wtd, lai)
        ax.scatter(x, y, s=170, color=COLORS[row["strategy_en"]], edgecolor="white", linewidth=1.6, zorder=3)
        text = (
            f"{row['strategy_en']}\n"
            f"ISA {isa*100:.1f}% | WTD {wtd*100:.1f}% | LAI {lai*100:.1f}%"
        )
        dx, dy = {
            "Target-first": (0.04, 0.075),
            "Maximum mitigation": (0.04, -0.065),
            "Cost-efficiency-first": (-0.05, 0.055),
            "Balanced compromise": (0.00, 0.10),
        }[row["strategy_en"]]
        ax.annotate(
            text, xy=(x, y), xytext=(x + dx, y + dy),
            fontsize=FS_ANNOT - 0.3,
            ha="left" if dx >= 0 else "right",
            va="center",
            arrowprops=dict(arrowstyle="-", color=GREY, lw=1.0),
            bbox=dict(boxstyle="round,pad=0.28", fc="white", ec=LIGHT_GREY, lw=0.9),
        )

    ax.text(
        0.50, -0.15,
        "Cost-efficiency-first is LAI-dominated, whereas maximum mitigation shifts strongly toward WTD expenditure.",
        ha="center", va="top", fontsize=FS_SMALL
    )


In [ ]:

# -----------------------------
# Figure writers
# -----------------------------
def make_main_figure(save_path):
    fig = plt.figure(figsize=(12.4, 9.4), facecolor="white")
    outer = GridSpec(
        2, 2, figure=fig,
        left=0.048, right=0.992, top=0.94, bottom=0.055,
        wspace=0.23, hspace=0.23
    )
    axA = fig.add_subplot(outer[0, 0])
    axB = fig.add_subplot(outer[0, 1])
    panel_a_strategy_landscape(axA)
    panel_b_area_ridgeline(axB)
    panel_c_attainment_intersections(fig, outer[1, 0])
    axD = fig.add_subplot(outer[1, 1])
    panel_d_cost_ternary(axD)

    fig.text(
        0.048, 0.987,
        "Fig. 1 | Comparative screening of four Pareto-based decision rules under a threshold of 0.35 in risk level change rate.",
        ha="left", va="top", fontsize=FS_FIG_TITLE, fontweight="bold"
    )
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

def make_panel_A(save_path):
    fig, ax = plt.subplots(figsize=(7.3, 5.8), facecolor="white")
    panel_a_strategy_landscape(ax)
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

def make_panel_B(save_path):
    fig, ax = plt.subplots(figsize=(6.9, 5.8), facecolor="white")
    panel_b_area_ridgeline(ax)
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

def make_panel_C(save_path):
    fig = plt.figure(figsize=(7.9, 5.9), facecolor="white")
    gs = GridSpec(1, 1, figure=fig, left=0.11, right=0.985, top=0.92, bottom=0.12)
    panel_c_attainment_intersections(fig, gs[0, 0])
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

def make_panel_D(save_path):
    fig, ax = plt.subplots(figsize=(6.9, 5.9), facecolor="white")
    panel_d_cost_ternary(ax)
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

def make_sup_province(save_path):
    prov = province.copy()
    prov = prov[prov["strategy_en"].isin(ORDER)]
    pivot = prov.pivot(index="province", columns="strategy_en", values="attain_rate")
    pivot = pivot.loc[pivot["Target-first"].sort_values(ascending=False).index]
    y = np.arange(len(pivot))

    # Width halved from the previous version, height unchanged
    fig, ax = plt.subplots(figsize=(4.1, 9.2), facecolor="white")
    style_ax(ax)
    mins = pivot.min(axis=1).values * 100
    maxs = pivot.max(axis=1).values * 100
    ax.hlines(y, mins, maxs, color=LIGHT_GREY, lw=3.0, zorder=1)

    offsets = np.linspace(-0.18, 0.18, len(ORDER))
    for off, strategy in zip(offsets, ORDER):
        ax.scatter(
            pivot[strategy].values * 100, y + off,
            s=46, color=COLORS[strategy], edgecolor="white", linewidth=0.85,
            label=strategy, zorder=3
        )

    ax.set_yticks(y)
    ax.set_yticklabels(pivot.index.tolist(), fontsize=10.2)
    ax.invert_yaxis()
    ax.set_xlabel("Threshold-reaching area fraction (%)", fontsize=FS_SMALL + 1)
    ax.set_xlim(0, 100)
    ax.set_title(
        "Supplementary Fig. 1 | Provincial heterogeneity in\nthreshold-reaching area fraction",
        fontsize=FS_SUP_TITLE, loc="left", pad=12
    )
    ax.legend(frameon=False, fontsize=9.5, ncol=1, loc="lower right", handletextpad=0.4, borderpad=0.2)
    fig.savefig(save_path, format="svg", bbox_inches="tight", pad_inches=0.015)
    plt.close(fig)

main_svg = PKG / "nature_v4_main_figure.svg"
sup_svg = PKG / "nature_v4_sup_fig1_province_rank_halfwidth.svg"
figA_svg = PKG / "nature_v4_figA_strategy_landscape.svg"
figB_svg = PKG / "nature_v4_figB_area_ridgeline.svg"
figC_svg = PKG / "nature_v4_figC_attainment_intersections.svg"
figD_svg = PKG / "nature_v4_figD_cost_ternary.svg"

make_main_figure(main_svg)
make_panel_A(figA_svg)
make_panel_B(figB_svg)
make_panel_C(figC_svg)
make_panel_D(figD_svg)
make_sup_province(sup_svg)

list(PKG.glob("*.svg"))


In [ ]:

md_text = """# Figure legend

**Fig. 1 | Comparative screening of four Pareto-based decision rules under a threshold of 0.35 in risk level change rate.**  
**a,** Strategy landscape. The x axis shows **Mean cost per 100 km² in 20 years** and the y axis shows **Risk level change rate**. Bubble area is proportional to the fraction of 100 km² areas reaching the 0.35 threshold. The annotation highlights that target-first and maximum mitigation reached the same threshold-oriented area fraction, but target-first did so at much lower mean cost.  
**b,** Area-level ridgeline distributions of **Risk level change rate** for the four decision rules. The dashed vertical line marks the threshold of 0.35. White dots indicate means and dark dots indicate medians.  
**c,** UpSet-style intersection analysis of threshold-reaching areas. Bars show the fraction of total analysed area belonging to each intersection, and the matrix below indicates which decision rules are involved.  
**d,** Ternary representation of cost composition among ISA, WTD and LAI. Point location indicates the relative contribution of the three cost components to the **Mean cost per 100 km² in 20 years**.  

Across 89,850 analysed 100 km² areas and 3,307,653 Pareto solutions, only 49,088 areas (54.63%) admitted at least one feasible solution reaching the 0.35 threshold in risk level change rate.
"""
(Path(PKG) / "nature_results_and_legends_v5.md").write_text(md_text, encoding="utf-8")

overall_en = overall.copy()
overall_en = overall_en.rename(columns={
    "strategy_en": "strategy",
    "attainment_rate_rr_ge_0_35": "threshold_reaching_area_fraction",
    "mean_RR": "mean_risk_level_change_rate",
    "median_RR": "median_risk_level_change_rate",
    "mean_y_pred_base": "mean_base_risk",
    "mean_y_pred_opt": "mean_optimised_risk",
    "mean_cost_total_million_usd": "mean_cost_per_100km2_in_20_years_million_usd",
    "median_cost_total_million_usd": "median_cost_per_100km2_in_20_years_million_usd",
    "total_cost_billion_usd": "total_cost_billion_usd",
    "mean_cost_ISA_million_usd": "mean_ISA_cost_million_usd",
    "mean_cost_WT_million_usd": "mean_WTD_cost_million_usd",
    "mean_cost_LAI_million_usd": "mean_LAI_cost_million_usd",
})
keep_cols = [
    "strategy", "n_grids", "threshold_reaching_area_fraction",
    "mean_risk_level_change_rate", "median_risk_level_change_rate",
    "mean_base_risk", "mean_optimised_risk",
    "mean_cost_per_100km2_in_20_years_million_usd",
    "median_cost_per_100km2_in_20_years_million_usd",
    "total_cost_billion_usd", "mean_ISA_cost_million_usd",
    "mean_WTD_cost_million_usd", "mean_LAI_cost_million_usd"
]
overall_en[keep_cols].to_csv(Path(PKG) / "nature_strategy_overall_summary_en_v5.csv", index=False)
print("Saved text and CSV.")
